# MCTS Kickstart Diagnostic

This notebook is the MCTS-side companion to `test_quick_diagnostic.ipynb`. It verifies that the current Monte Carlo Tree Search can score service-network actions with the updated `ServiceGraph.fulfill_demands(...)` MILP, while keeping the default run small enough for iteration.

In [ ]:
from pathlib import Path
import random
import sys
import time

import numpy as np
import pandas as pd

# Find the project root (the folder that contains src/cma) by walking upward
# from the current working directory. This works whether the notebook is run
# from notebooks/ or from the repo root. If you have run `pip install -e .`,
# this sys.path setup is harmless and not strictly needed.
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'cma').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cma.data_reader import (
    read_vessel_class_data,
    read_port_data,
    read_sailing_distance_data,
    read_demand_with_transit_time,
    read_cnc_proforma_data,
)
from cma.mcts import MonteCarloTree
from cma.port import PortGraph
from cma.servicegraph import ServiceGraph

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

SEED = 7
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
#!pip install shapely

## 1. Load Current Data

In [ ]:
vesselpool = read_vessel_class_data()
portpool_main, portpool_dmd = read_port_data()
dist_matrix = read_sailing_distance_data(portpool_main)
demand_matrix, transit_time_matrix = read_demand_with_transit_time(portpool_main)

proforma = read_cnc_proforma_data(portpool_main, vesselpool, dist_matrix=dist_matrix)
all_service_lines = proforma['lines']

print(f'Loaded {len(all_service_lines)} proforma service lines')
print(f'Loaded {portpool_main.get_number_of_ports()} ports')
print(f'Loaded {int((np.asarray(demand_matrix) > 0).sum())} positive OD demands')

Loaded 31 proforma service lines
Loaded 182 ports
Loaded 741 positive OD demands


## 2. Build A Smoke-Test MCTS Problem

In [ ]:
SMOKE_LINE_COUNT = 8
SMOKE_OD_COUNT = 20

full_portgraph = PortGraph(
    portpool_main,
    dist_matrix,
    demand_matrix,
    mat_transit_time=transit_time_matrix,
    filter_by_demand=False,
)

line_count = min(SMOKE_LINE_COUNT, len(all_service_lines))
while True:
    service_lines = all_service_lines[:line_count]
    servicegraph = ServiceGraph(service_lines)
    _, feasible_actions = servicegraph.get_feasible_actions(full_portgraph)
    paths_dict = servicegraph.get_all_paths(full_portgraph, full_portgraph.filtered_by_transship_capacity())
    connected_positive = [
        (od, demand, paths)
        for od, demand, paths in zip(
            paths_dict['od_pairs'],
            paths_dict['od_pairs_demand'],
            paths_dict['od_pairs_path'],
        )
        if demand > 0 and len(paths) > 0
    ]
    if feasible_actions and connected_positive:
        break
    line_count += 2
    if line_count > len(all_service_lines):
        raise RuntimeError('Could not find a service-line subset with feasible MCTS actions and connected demand.')

demand_smoke = np.zeros_like(np.asarray(demand_matrix, dtype=float))
for (origin_idx, dest_idx), demand, _ in connected_positive[:SMOKE_OD_COUNT]:
    demand_smoke[origin_idx, dest_idx] = demand

portgraph = PortGraph(
    portpool_main,
    dist_matrix,
    demand_smoke,
    mat_transit_time=transit_time_matrix,
    filter_by_demand=False,
)

smoke_paths = servicegraph.get_all_paths(portgraph, portgraph.filtered_by_transship_capacity())
_, smoke_actions = servicegraph.get_feasible_actions(portgraph)

print(f'Using {len(service_lines)} service lines')
print(f'Using {len(smoke_paths["od_pairs"])} connected smoke OD pairs')
print(f'Root feasible MCTS actions: {len(smoke_actions)}')
print(f'Smoke weekly demand: {demand_smoke.sum():,.0f} TEU')

Using 8 service lines
Using 20 connected smoke OD pairs
Root feasible MCTS actions: 7235
Smoke weekly demand: 21,260 TEU


## 3. MILP Settings Used By MCTS

In [ ]:
USE_ACCURATE_DISCRETE_SPEED = False

WEEK_LEVELS = [1, 2, 3, 4, 5, 6, 7]

MCTS_MILP_TUNEPARAMS = {
    'turnon-transship_shipclass_restriction': 0,
    'turnon-vessel_speed_optimization': 0 if USE_ACCURATE_DISCRETE_SPEED else 1,
    'turnon-tight_line_capacity_linearization': 1,
    'turnon-port_operations_constraint': 1,
    'turnon-transit_time_penalty': 1,
    'turnon-demand_fulfillment_cap': 1,
    'ctrparam-kts_buffer': 0,
    'ctrparam-transship_A': 100,
    'ctrparam-speed_soft_cap_kts': 16.5,
    'ctrparam-speed_penalty_multiplier': 2.0,
    'ctrparam-transit_penalty_multiplier': 1000.0,
    'ctrparam-buffer_penalty_below_15pct': 1000.0,
    'ctrparam-buffer_penalty_above_30pct': 5000.0,
    'unfulfilled_demand_penalty': 1e6,
    'BigM-transship': 10000,
    'BigM-n_ships': 10,
    'BigM-saildays': 100,
    'BigM-line_capacity': 30000,
    'BigM-portcall_cost': 1e7,
    'turnon-schedule_adherence': 0,
    'schedule_buffer_hrs': 120.0,
    'solver-MIPGap': 0.10,
    'solver-TimeLimit': 90,
    'solver-MIPFocus': 1,
    'solver-verbose': False,
}

MCTS_MILP_TUNEPARAMS

{'turnon-transship_shipclass_restriction': 0,
 'turnon-vessel_speed_optimization': 1,
 'turnon-tight_line_capacity_linearization': 1,
 'turnon-port_operations_constraint': 1,
 'turnon-transit_time_penalty': 1,
 'turnon-demand_fulfillment_cap': 1,
 'ctrparam-kts_buffer': 0,
 'ctrparam-transship_A': 100,
 'ctrparam-speed_soft_cap_kts': 16.5,
 'ctrparam-speed_penalty_multiplier': 2.0,
 'ctrparam-transit_penalty_multiplier': 1000.0,
 'ctrparam-buffer_penalty_below_15pct': 1000.0,
 'ctrparam-buffer_penalty_above_30pct': 5000.0,
 'unfulfilled_demand_penalty': 1000000.0,
 'BigM-transship': 10000,
 'BigM-n_ships': 10,
 'BigM-saildays': 100,
 'BigM-line_capacity': 30000,
 'BigM-portcall_cost': 10000000.0,
 'turnon-schedule_adherence': 0,
 'schedule_buffer_hrs': 120.0,
 'solver-MIPGap': 0.1,
 'solver-TimeLimit': 90,
 'solver-MIPFocus': 1,
 'solver-verbose': False}

## 4. Baseline Root Evaluation

In [ ]:
RUN_BASELINE_SOLVE = True

if RUN_BASELINE_SOLVE:
    start_time = time.time()
    baseline_solution = servicegraph.solve_approximated(
        portgraph,
        vesselpool,
        min_cost=True,
        week_levels=WEEK_LEVELS,
        tuneparams_2=MCTS_MILP_TUNEPARAMS,
    )
    elapsed = time.time() - start_time
    print(f'Baseline solve time: {elapsed:.2f}s')
    print(f'Solver status: {baseline_solution.get("solver_status")} ({baseline_solution.get("solver_name")})')
    print(f'Root cost: {servicegraph.total_cost():,.2f}')
    print(f'Buffer >30% lines: {baseline_solution.get("kpi_lines_buffer_above_30", 0)}')
    print(f'Buffer penalty: {baseline_solution.get("buffer penalty cost", 0):,.2f}')
else:
    baseline_solution = None
    print('Baseline solve skipped.')

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2837476
Academic license 2837476 - for non-commercial use only - registered to an___@u.nus.edu
Baseline solve time: 2.69s
Solver status: infeasible (GUROBI)
Root cost: inf
Buffer >30% lines: 0
Buffer penalty: 0.00


## 5. Run A Small MCTS Search

In [ ]:
MCTS_EPOCHS = 6

tree = MonteCarloTree(
    servicegraph=servicegraph,
    portgraph=portgraph,
    vesselpool=vesselpool,
    discount_fac=0.5,
    valid_weight_proportion=0.74,
    max_depth=2,
    c_param=1e-2,
    min_cost=True,
    week_levels=WEEK_LEVELS,
    milp_tuneparams=MCTS_MILP_TUNEPARAMS,
)

start_time = time.time()
tree.run(MCTS_EPOCHS, display=True)
elapsed = time.time() - start_time

print(f'MCTS wall time: {elapsed:.2f}s')
print(f'Recorder: {tree.recorder()}')
print(f'Total tree nodes: {tree.total_number_of_nodes()}')

Select: <cma.mcts.MonteCarloTreeSearchNode object at 0x00000208B91667B0> rollout
Select: <cma.mcts.MonteCarloTreeSearchNode object at 0x00000208B91667B0> expand


KeyboardInterrupt: 

## 6. Inspect Best Action

In [ ]:
best_node = tree.get_best_node()
best_actions = best_node.trace_actions()

print(f'Root visits: {tree.root_node.number_of_visits}')
print(f'Root cost: {tree.root_node.graph.total_cost():,.2f}')
print(f'Best depth: {best_node.get_depth()}')
print(f'Best cost: {best_node.graph.total_cost():,.2f}')

if best_actions:
    tree.display_best_node(portgraph)
else:
    print('No improving child was selected in this smoke run. Increase MCTS_EPOCHS or valid_weight_proportion for a deeper search.')

Root visits: 4
Root cost: 10,906,128,254.96
Best depth: 0
Best cost: 10,906,128,254.96
No improving child was selected in this smoke run. Increase MCTS_EPOCHS or valid_weight_proportion for a deeper search.


In [ ]:
ACTIONS_TO_SHOW = 8

child_rows = []
for idx, (action, child) in enumerate(zip(tree.root_node.borns, tree.root_node.children), start=1):
    child_rows.append({
        'child': idx,
        'visits': child.number_of_visits,
        'depth': child.get_depth(),
        'cost': child.graph.total_cost(),
        'reward': child.current_state_reward(),
        'action': action.explain(portgraph, tree.root_node.graph, idx).replace('\n', ' '),
    })

children_df = pd.DataFrame(child_rows)
if len(children_df) > 0:
    children_df = children_df.sort_values(['visits', 'cost'], ascending=[False, True]).head(ACTIONS_TO_SHOW)

print(f'Invalid candidate actions skipped at root: {len(tree.root_node.invalid_actions)}')
children_df

Invalid candidate actions skipped at root: 0


,child,visits,depth,cost,reward,action
0,1,3,1,1.090763e+10,9.167896e-11,"Action 1: For the #5 line ""CMS2CNC -- [CNS..."


## Scaling Notes

For a stronger MCTS run, increase `MCTS_EPOCHS`, `max_depth`, and `valid_weight_proportion`. For a truer but slower MILP oracle, set `USE_ACCURATE_DISCRETE_SPEED = True` and increase `solver-TimeLimit`; keep `solver-MIPGap` unchanged when you want comparable approximate-search behavior.